# EUR/USD — komparativna analiza modela mašinskog učenja

Glavni notebook eksperimenta (predviđanje binarnog smjera kretanja kursa EUR/USD). Ćelije se izvršavaju **redom, odozgo na dole, jednom**.
Sva logika živi u `src/`; ovdje se samo pokreće eksperiment, pregledaju rezultati i crtaju slike za rad.

1. Podešavanje: biblioteke, `config.yaml` i FRED API ključ
2. Pun eksperiment (23 godine): pokretanje i pregled rezultata
3. Kraći eksperiment (2 godine): pokretanje i pregled rezultata
4. Koeficijenti logističke regresije
5. Slike za rad (Slike 4–9)

Najsporije su ćelije koje pozivaju `run_experiment()` (pretraga i trening LSTM mreže); sve ostale se izvršavaju odmah.
Rezultati se čuvaju u `results/` (`summary_<tag>.json` je izvor za tabele i slike u radu).

In [1]:
import getpass
import json
import sys
from pathlib import Path

# Korijen projekta: notebook se može pokrenuti iz notebooks/ ili iz korijena projekta
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels
import tensorflow as tf
import xgboost
import lightgbm
from sklearn.metrics import accuracy_score

from src.figures import load as load_summary, make_all_figures
from src.models.sklearn_models import make_pipeline
from src.pipelines import run_experiment
from src.split import chronological_split, make_xy
from src.utils import load_config, resolve_path

print("Verzije biblioteka (dokumentovati u Poglavlju 3):")
for name, mod in [("pandas", pd), ("numpy", np), ("sklearn", sklearn),
                  ("xgboost", xgboost), ("lightgbm", lightgbm),
                  ("statsmodels", statsmodels), ("scipy", scipy),
                  ("tensorflow", tf)]:
    print(f"  {name}: {mod.__version__}")

Verzije biblioteka (dokumentovati u Poglavlju 3):
  pandas: 3.0.3
  numpy: 1.26.4
  sklearn: 1.5.2
  xgboost: 3.3.0
  lightgbm: 4.7.0
  statsmodels: 0.14.6
  scipy: 1.13.1
  tensorflow: 2.16.1


## 1. Konfiguracija i FRED API ključ

In [2]:
cfg = load_config()
print(f"Period (pun eksperiment): {cfg['data']['start_date']} -> {cfg['data']['end_date']}")
print(f"Period (kratak eksperiment): {cfg['data']['short_start_date']} -> {cfg['data']['short_end_date']}")
print("Kasnjenje FRED serija (dani):", cfg["fred"]["series_lag_days"])
print("Reprezentativno LSTM pokretanje:", cfg["lstm"]["representative_run_index"])
print("Statistika:", cfg["statistics"])

Period (pun eksperiment): 2003-01-01 -> 2026-01-01
Period (kratak eksperiment): 2024-01-01 -> 2026-01-01
Kasnjenje FRED serija (dani): {'CPIAUCSL': 0, 'UNRATE': 0, 'DFF': 1, 'ECBDFR': 1}
Reprezentativno LSTM pokretanje: 0
Statistika: {'block_length': 10}


In [3]:
# Ključ se unosi ručno i postoji samo u memoriji sesije (ne upisuje se u kod ni u config.yaml).
FRED_API_KEY = getpass.getpass("Unesi FRED API kljuc: ")

## 2. Pun eksperiment (23 godine)

`run_experiment()` preuzima podatke (EUR/USD i FRED/ALFRED), gradi karakteristike, testira stacionarnost (ADF), dijeli podatke hronološki (sa razmakom `gap`),
trenira referentne modele i četiri klasična modela (sa early stopping-om i kriterijumom rangiranja) i LSTM (sa pretragom arhitekture),
računa statističke testove i sve čuva u `results/`.

In [4]:
results_full = run_experiment(cfg, FRED_API_KEY, use_short_period=False)

print(f"Eksperiment: {results_full['tag']}")
print(f"Uzoraka: {results_full['dataset_summary']['n_samples']}")
print(f"Period: {results_full['dataset_summary']['date_start']} -> {results_full['dataset_summary']['date_end']}")

Eksperiment: full_23y
Uzoraka: 5477
Period: 2004-11-18 -> 2025-12-30


### 2.1 Tabela rezultata i podjela podataka

In [5]:
results_table_full = pd.DataFrame(results_full["results_table"]).set_index("model")
print("=== KOMPLETNA TABELA REZULTATA ===")
print(results_table_full.round(4))
print()
print("=== PODJELA PODATAKA ===")
print("Gap dani:", results_full["split_info"]["gap_days"])
print("Ociscenih redova (purging):", results_full["split_info"]["purged_rows"])
for name in ["train", "val", "test"]:
    print(f"  {name}: {results_full['split_info'][name]}")

=== KOMPLETNA TABELA REZULTATA ===
                      n_samples  accuracy  precision  recall      f1   TN  \
model                                                                       
baseline_persistence        823    0.5261     0.5125  0.5125  0.5125  228   
baseline_majority           823    0.5140     0.0000  0.0000  0.0000  423   
svm                         823    0.5103     0.4946  0.3450  0.4065  282   
lstm                        823    0.5030     0.4795  0.2625  0.3393  309   
logreg                      823    0.4933     0.4653  0.2850  0.3535  292   
baseline_random             823    0.4897     0.4762  0.5000  0.4878  203   
lightgbm                    823    0.4800     0.4573  0.3750  0.4121  245   
baseline_ar1                823    0.4775     0.4631  0.4700  0.4665  205   
xgboost                     823    0.4763     0.4553  0.3950  0.4230  234   

                       FP   FN   TP  
model                                
baseline_persistence  195  195  205  
bas

### 2.2 Test stacionarnosti (ADF)

In [6]:
print(pd.DataFrame(results_full["stationarity_table"]).to_string(index=False))

                 feature  adf_statistic  p_value  n_obs  used_lag  critical_1%  critical_5%  critical_10%  stationary_at_5%
                  Return       -13.6619 0.000000   5446        30      -3.4316      -2.8621       -2.5671              True
               SMA_ratio       -11.4516 0.000000   5444        32      -3.4316      -2.8621       -2.5671              True
           Volatility_5D        -6.7259 0.000000   5449        27      -3.4316      -2.8621       -2.5671              True
            Momentum_10D       -10.4694 0.000000   5443        33      -3.4316      -2.8621       -2.5671              True
             Daily_Range        -6.5321 0.000000   5447        29      -3.4316      -2.8621       -2.5671              True
                  RSI_14       -10.6823 0.000000   5443        33      -3.4316      -2.8621       -2.5671              True
                 CPI_YoY        -3.2302 0.018302   5452        24      -3.4315      -2.8621       -2.5671              True
        

### 2.3 Foldovi unakrsne validacije

In [7]:
for fold in results_full["cv_fold_info"]:
    print(fold)

{'fold': 1, 'train_n': 641, 'train_start': '2004-11-18', 'train_end': '2007-05-04', 'gap_days': 1, 'val_n': 638, 'val_start': '2007-05-08', 'val_end': '2009-11-16'}
{'fold': 2, 'train_n': 1279, 'train_start': '2004-11-18', 'train_end': '2009-11-13', 'gap_days': 1, 'val_n': 638, 'val_start': '2009-11-17', 'val_end': '2012-04-26'}
{'fold': 3, 'train_n': 1917, 'train_start': '2004-11-18', 'train_end': '2012-04-25', 'gap_days': 1, 'val_n': 638, 'val_start': '2012-04-27', 'val_end': '2014-10-09'}
{'fold': 4, 'train_n': 2555, 'train_start': '2004-11-18', 'train_end': '2014-10-08', 'gap_days': 1, 'val_n': 638, 'val_start': '2014-10-10', 'val_end': '2017-03-21'}
{'fold': 5, 'train_n': 3193, 'train_start': '2004-11-18', 'train_end': '2017-03-20', 'gap_days': 1, 'val_n': 638, 'val_start': '2017-03-22', 'val_end': '2019-09-04'}


### 2.4 Hiperparametri: šta je našla pretraga, kriterijum rangiranja i šta je stvarno korišćeno

In [8]:
print("=== best_hyperparameters (sta je Grid Search pronasao) ===")
print(json.dumps(results_full["best_hyperparameters"], indent=2))
print()
print("=== KRITERIJUM RANGIRANJA (fabricka vs. grid-search, po validaciji) ===")
print(json.dumps(results_full["model_selection"], indent=2))
print()
print("=== final_params_used (STVARNO koriscene vrijednosti - koristi OVO u tekstu) ===")
print(json.dumps(results_full["final_params_used"], indent=2))
print()
print("=== early_stopping_info (samo XGBoost/LightGBM) ===")
print(json.dumps(results_full["early_stopping_info"], indent=2))

=== best_hyperparameters (sta je Grid Search pronasao) ===
{
  "logreg": {
    "clf__C": 10
  },
  "svm": {
    "clf__C": 0.1,
    "clf__gamma": "scale",
    "clf__kernel": "rbf"
  },
  "xgboost": {
    "clf__learning_rate": 0.01,
    "clf__max_depth": 3,
    "clf__n_estimators": 35
  },
  "lightgbm": {
    "clf__learning_rate": 0.01,
    "clf__max_depth": 3,
    "clf__n_estimators": 84
  },
  "lstm": {
    "sequence_length": 20,
    "n_units": 16,
    "n_layers": 2,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "batch_size": 32,
    "threshold": 0.5000000000000002,
    "source": "grid_search"
  }
}

=== KRITERIJUM RANGIRANJA (fabricka vs. grid-search, po validaciji) ===
{
  "logreg": {
    "default_val_accuracy": 0.4988,
    "grid_search_val_accuracy": 0.4988,
    "grid_search_cv_accuracy": 0.5157,
    "grid_search_cv_std": 0.0164,
    "selected": "grid_search"
  },
  "svm": {
    "default_val_accuracy": 0.4976,
    "grid_search_val_accuracy": 0.5,
    "grid_search_cv_accuracy":

### 2.5 Tačnost prije i poslije optimizacije

In [9]:
print(pd.DataFrame(results_full["optimization_comparison"]))
print()
print("Broj poboljsanih modela:", results_full["n_models_improved"])

      model  accuracy_before_optimization  accuracy_after_optimization  \
0    logreg                        0.4933                       0.4933   
1       svm                        0.4921                       0.5103   
2   xgboost                        0.4763                       0.4763   
3  lightgbm                        0.4812                       0.4800   
4      lstm                        0.5072                       0.4846   

   improved  std_before  std_after  
0     False         NaN        NaN  
1      True         NaN        NaN  
2     False         NaN        NaN  
3     False         NaN        NaN  
4     False      0.0132     0.0152  

Broj poboljsanih modela: 1/5


### 2.6 Generalizacioni jaz

In [10]:
print(pd.DataFrame(results_full["generalization_table"]).to_string(index=False))

   model  trainval_accuracy  test_accuracy  trainval_test_gap  selection_train_accuracy  selection_val_accuracy
 xgboost             0.9791         0.4763             0.5028                    0.9851                  0.5256
lightgbm             0.5626         0.4800             0.0826                    0.5684                  0.5232
  logreg             0.5258         0.4933             0.0325                    0.5261                  0.4988
     svm             0.5337         0.5103             0.0234                    0.5297                  0.5000
    lstm             0.5070         0.5030             0.0040                    0.5098                  0.5171


### 2.7 Tačnost kroz tri potperioda testnog skupa

In [11]:
for name in ["logreg", "svm", "xgboost", "lightgbm", "lstm",
             "baseline_majority", "baseline_persistence", "baseline_random", "baseline_ar1"]:
    df = pd.DataFrame(results_full["subperiod_accuracy"][name])
    print(f"\n{name}:")
    print(df[["period", "start", "end", "n", "accuracy"]].to_string(index=False))


logreg:
 period      start        end   n  accuracy
      1 2022-10-31 2023-11-16 274    0.4854
      2 2023-11-17 2024-12-04 274    0.4964
      3 2024-12-05 2025-12-30 275    0.4982

svm:
 period      start        end   n  accuracy
      1 2022-10-31 2023-11-16 274    0.4818
      2 2023-11-17 2024-12-04 274    0.5036
      3 2024-12-05 2025-12-30 275    0.5455

xgboost:
 period      start        end   n  accuracy
      1 2022-10-31 2023-11-16 274    0.4818
      2 2023-11-17 2024-12-04 274    0.4562
      3 2024-12-05 2025-12-30 275    0.4909

lightgbm:
 period      start        end   n  accuracy
      1 2022-10-31 2023-11-16 274    0.4891
      2 2023-11-17 2024-12-04 274    0.4781
      3 2024-12-05 2025-12-30 275    0.4727

lstm:
 period      start        end   n  accuracy
      1 2022-10-31 2023-11-16 274    0.4927
      2 2023-11-17 2024-12-04 274    0.4818
      3 2024-12-05 2025-12-30 275    0.5345

baseline_majority:
 period      start        end   n  accuracy
      1 2022-

### 2.8 Intervali pouzdanosti i McNemarov test

In [12]:
print("=== BOOTSTRAP 95% CI ZA TACNOST ===")
for name, val in results_full["statistical_tests"].items():
    if name.startswith("bootstrap_ci_"):
        print(f"{name.replace('bootstrap_ci_',''):10s}: {val['point_estimate']:.4f}  "
              f"[{val['ci_lower']:.4f}, {val['ci_upper']:.4f}]")

print()
print("=== McNEMAR: NAJBOLJI MODEL vs. BASELINE-I ===")
for name, val in results_full["statistical_tests"].items():
    if name.startswith("mcnemar_") and "baseline" in name:
        sig = "ZNACAJNO (p<0.05)" if val["p_value"] < 0.05 else "nije znacajno"
        print(f"{name}: p={val['p_value']:.4f} -> {sig}  (n_01={val['n_01']}, n_10={val['n_10']})")

print()
print("=== McNEMAR: KOMPLEKSNI vs. LINEARNI MODELI (direktan test H1) ===")
for name, val in results_full["statistical_tests"].items():
    if "_vs_logreg" in name or "_vs_svm" in name:
        sig = "ZNACAJNO (p<0.05)" if val["p_value"] < 0.05 else "nije znacajno"
        print(f"{name}: p={val['p_value']:.4f} -> {sig}")

=== BOOTSTRAP 95% CI ZA TACNOST ===
logreg    : 0.4933  [0.4593, 0.5273]
svm       : 0.5103  [0.4751, 0.5443]
xgboost   : 0.4763  [0.4423, 0.5103]
lightgbm  : 0.4800  [0.4471, 0.5140]
lstm      : 0.5030  [0.4678, 0.5359]
baseline_majority: 0.5140  [0.4787, 0.5468]
baseline_random: 0.4897  [0.4532, 0.5237]
baseline_persistence: 0.5261  [0.4921, 0.5601]
baseline_ar1: 0.4775  [0.4435, 0.5128]

=== McNEMAR: NAJBOLJI MODEL vs. BASELINE-I ===
mcnemar_logreg_vs_baseline_majority: p=0.3067 -> nije znacajno  (n_01=131, n_10=114)
mcnemar_logreg_vs_baseline_random: p=0.9204 -> nije znacajno  (n_01=199, n_10=202)
mcnemar_logreg_vs_baseline_persistence: p=0.2749 -> nije znacajno  (n_01=297, n_10=270)
mcnemar_logreg_vs_baseline_ar1: p=0.4433 -> nije znacajno  (n_01=116, n_10=129)
mcnemar_svm_vs_baseline_majority: p=0.9047 -> nije znacajno  (n_01=141, n_10=138)
mcnemar_svm_vs_baseline_random: p=0.4355 -> nije znacajno  (n_01=202, n_10=219)
mcnemar_svm_vs_baseline_persistence: p=0.6066 -> nije znacajn

## 3. Kraći eksperiment (2 godine)

Ista funkcija `run_experiment()`, samo drugi granični datumi. Kraći period leži unutar test perioda dužeg eksperimenta,
pa ovo nije nezavisan eksperiment nego dopunska provjera osjetljivosti.

In [13]:
results_short = run_experiment(cfg, FRED_API_KEY, use_short_period=True)

print(f"Eksperiment: {results_short['tag']}")
print(f"Uzoraka: {results_short['dataset_summary']['n_samples']}")
print(f"Period: {results_short['dataset_summary']['date_start']} -> {results_short['dataset_summary']['date_end']}")
print(f"Split: {results_short['split_info']}")

Eksperiment: short_2y
Uzoraka: 265
Period: 2024-12-19 -> 2025-12-30
Split: {'train': {'n': 184, 'start': '2024-12-19', 'end': '2025-09-05'}, 'val': {'n': 38, 'start': '2025-09-09', 'end': '2025-10-30'}, 'test': {'n': 41, 'start': '2025-11-03', 'end': '2025-12-30'}, 'gap_days': 1, 'purged_rows': {'train': 1, 'val': 1}}


### 3.1 Tabela rezultata, kriterijum rangiranja i generalizacioni jaz

In [14]:
print("=== KOMPLETNA TABELA REZULTATA (2-god.) ===")
print(pd.DataFrame(results_short["results_table"]).set_index("model")[["accuracy", "n_samples"]].round(4))
print()
print("=== KRITERIJUM RANGIRANJA (2-god.) ===")
print(json.dumps(results_short["model_selection"], indent=2))
print()
print("=== GENERALIZACIONI JAZ - H3 (2-god.) ===")
print(pd.DataFrame(results_short["generalization_table"]).to_string(index=False))

=== KOMPLETNA TABELA REZULTATA (2-god.) ===
                      accuracy  n_samples
model                                    
baseline_random         0.6098         41
lstm                    0.5610         41
baseline_persistence    0.5610         41
logreg                  0.5366         41
baseline_majority       0.5122         41
svm                     0.4878         41
baseline_ar1            0.4878         41
lightgbm                0.4634         41
xgboost                 0.4146         41

=== KRITERIJUM RANGIRANJA (2-god.) ===
{
  "logreg": {
    "default_val_accuracy": 0.5789,
    "grid_search_val_accuracy": 0.5526,
    "grid_search_cv_accuracy": 0.5733,
    "grid_search_cv_std": 0.0573,
    "selected": "default"
  },
  "svm": {
    "default_val_accuracy": 0.5263,
    "grid_search_val_accuracy": 0.5,
    "grid_search_cv_accuracy": 0.5267,
    "grid_search_cv_std": 0.0879,
    "selected": "default"
  },
  "xgboost": {
    "default_val_accuracy": 0.6053,
    "grid_search_va

### 3.2 Intervali pouzdanosti

In [15]:
print("=== BOOTSTRAP 95% CI (2-god.) ===")
for name, val in results_short["statistical_tests"].items():
    if name.startswith("bootstrap_ci_"):
        print(f"{name.replace('bootstrap_ci_',''):10s}: {val['point_estimate']:.4f}  "
              f"[{val['ci_lower']:.4f}, {val['ci_upper']:.4f}]")

=== BOOTSTRAP 95% CI (2-god.) ===
logreg    : 0.5366  [0.3902, 0.6829]
svm       : 0.4878  [0.3415, 0.6341]
xgboost   : 0.4146  [0.2683, 0.5610]
lightgbm  : 0.4634  [0.3171, 0.6098]
lstm      : 0.5610  [0.3902, 0.7073]
baseline_majority: 0.5122  [0.3659, 0.6585]
baseline_random: 0.6098  [0.4634, 0.7561]
baseline_persistence: 0.5610  [0.4146, 0.7073]
baseline_ar1: 0.4878  [0.3415, 0.6341]


## 4. Koeficijenti logističke regresije

Koeficijenti **istog konačnog modela** koji je dao rezultate u tabelama: isti podaci (`data/processed`), ista podjela, isti parametri (`final_params_used`),
fit na `D_trainval`. Funkcija provjerava da se tačnosti poklapaju sa generalizacionom tabelom iz rezultata (u suprotnom prekida izvršavanje).
Mora se pokrenuti poslije oba eksperimenta.

In [16]:
def logreg_koeficijenti(results: dict, tag: str, cfg: dict, tol: float = 6e-4) -> pd.DataFrame:
    """Koeficijenti ISTOG konacnog LogReg modela koji je dao rezultate u tabelama:
    isti podaci (data/processed), ista podjela, isti parametri (final_params_used),
    fit na D_trainval. Provjerava se da tacnosti odgovaraju summary JSON-u."""
    model_df = pd.read_csv(resolve_path(cfg["paths"]["data_processed"], f"model_df_{tag}.csv"),
                           index_col=0, parse_dates=True)
    feature_cols = results["dataset_summary"]["feature_columns"]
    train_df, val_df, test_df, _ = chronological_split(
        model_df, cfg["split"]["train_fraction"], cfg["split"]["val_fraction"],
        gap=cfg["split"].get("gap", 1))
    X_tv, y_tv = make_xy(pd.concat([train_df, val_df]), feature_cols)
    X_te, y_te = make_xy(test_df, feature_cols)

    pipe = make_pipeline("logreg", cfg["random_seed"])
    pipe.set_params(**results["final_params_used"]["logreg"])
    pipe.fit(X_tv, y_tv)

    # Provjera: ovo MORA biti isti model kao u generalization_table
    ref = {r["model"]: r for r in results["generalization_table"]}["logreg"]
    tv_acc = accuracy_score(y_tv, pipe.predict(X_tv))
    te_acc = accuracy_score(y_te, pipe.predict(X_te))
    assert abs(tv_acc - ref["trainval_accuracy"]) < tol, (tv_acc, ref["trainval_accuracy"])
    assert abs(te_acc - ref["test_accuracy"]) < tol, (te_acc, ref["test_accuracy"])

    clf = pipe.named_steps["clf"]
    print(f"Eksperiment: {tag} | D_trainval: {len(y_tv)} redova | C = {clf.C}")
    print(f"Tacnost D_trainval = {tv_acc:.4f}, D_test = {te_acc:.4f}  (isto kao u tabeli)")
    print(f"Intercept (β0): {clf.intercept_[0]:.10f}")
    coef = pd.DataFrame({"feature": feature_cols, "coefficient": clf.coef_[0]})
    coef = coef.reindex(coef["coefficient"].abs().sort_values(ascending=False).index)
    print("Koeficijenti su za STANDARDIZOVANA obiljezja (StandardScaler u pipeline-u).")
    print(coef.to_string(index=False))
    return coef


coef_full = logreg_koeficijenti(results_full, "full_23y", cfg)
print()
coef_short = logreg_koeficijenti(results_short, "short_2y", cfg)

Eksperiment: full_23y | D_trainval: 4652 redova | C = 10
Tacnost D_trainval = 0.5258, D_test = 0.4933  (isto kao u tabeli)
Intercept (β0): -0.0104451908
Koeficijenti su za STANDARDIZOVANA obiljezja (StandardScaler u pipeline-u).
                 feature  coefficient
            Momentum_10D    -0.135607
               SMA_ratio     0.083646
                 CPI_YoY    -0.057389
                  Return    -0.056501
             UNRATE_Diff     0.032514
Rate_Differential_Change     0.019728
                  RSI_14    -0.001791
           Volatility_5D     0.001438
             Daily_Range    -0.000185

Eksperiment: short_2y | D_trainval: 222 redova | C = 1.0
Tacnost D_trainval = 0.6126, D_test = 0.5366  (isto kao u tabeli)
Intercept (β0): -0.0368884206
Koeficijenti su za STANDARDIZOVANA obiljezja (StandardScaler u pipeline-u).
                 feature  coefficient
                  RSI_14    -0.485798
               SMA_ratio     0.263609
           Volatility_5D     0.203467
         

## 5. Slike za rad (Slike 4–9)

Ne trenira ništa: čita sačuvane `results/summary_<tag>.json` i crta ono što je eksperiment sačuvao (`src/figures.py`).
Slike se upisuju u `results/figures_rad/`.

In [17]:
RESULTS_DIR = ROOT / cfg["paths"]["results"]
summary_full = load_summary(RESULTS_DIR / f"summary_{results_full['tag']}.json")
summary_short = load_summary(RESULTS_DIR / f"summary_{results_short['tag']}.json")

for path in make_all_figures(summary_full, summary_short, RESULTS_DIR / "figures_rad"):
    print("Sacuvano:", Path(path).relative_to(ROOT))

Sacuvano: results\figures_rad\slika4_optimizacija.png
Sacuvano: results\figures_rad\slika5_rangirana_tacnost.png
Sacuvano: results\figures_rad\slika6_matrice_konfuzije_puni.png
Sacuvano: results\figures_rad\slika7_matrice_konfuzije_kraci.png
Sacuvano: results\figures_rad\slika8_generalizacioni_jaz.png
Sacuvano: results\figures_rad\slika9_potperiodi.png
